In [1]:
#Dataset
from datasets import load_dataset

imdb_dataset = load_dataset(
    "csv", data_files={"train": "train.csv", "eval": "eval.csv"}
)

imdb_dataset

imdb_dataset["train"][645]

{'index': 2981,
 'review': 'Brilliant execution in displaying once and for all, this time in the venue of politics, of how "good intentions do actually pave the road to hell". Excellent!',
 'label': 1}

In [2]:
#Tokeniser
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_function(batch):
    return tokenizer(batch["review"], padding=True, truncation=True)


tokenized_imdb_dataset = imdb_dataset.map(tokenize_function, batched=True)

tokenized_imdb_dataset

DatasetDict({
    train: Dataset({
        features: ['index', 'review', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2000
    })
    eval: Dataset({
        features: ['index', 'review', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 500
    })
})

In [3]:
import os

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [4]:
#Trainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="tmp_trainer",
    eval_strategy="epoch",
    logging_steps=5,
)

import evaluate

accuracy = evaluate.load("accuracy")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [5]:
from transformers import Trainer


def make_trainer(model):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_imdb_dataset["train"],
        eval_dataset=tokenized_imdb_dataset["eval"],
        compute_metrics=compute_metrics,
    )
    return trainer

In [6]:
#Full fine-tuning
from transformers import AutoModelForSequenceClassification

pretrained_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)
pretrained_model


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [8]:
#Task 4.01: Counting the number of trainable parameters
def num_trainable_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Test — should print 66,955,010
print(num_trainable_parameters(pretrained_model))


66955010


In [ ]:
finetuned_trainer = make_trainer(pretrained_model)
finetuned_trainer.train()
finetuned_trainer.save_model("finetuned")
finetuned_model = AutoModelForSequenceClassification.from_pretrained("finetuned")

def train(model):
    print("Trainable parameters:", num_trainable_parameters(model))
    make_trainer(model).train()
    return model

def evaluate_model(model):
    return make_trainer(model).evaluate()

C:\Users\Manvi\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


In [ ]:
#Task 4.02: Head-tuning
def make_headtuned_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2
    )
    for param in model.parameters():
        param.requires_grad = False
    for param in model.pre_classifier.parameters():
        param.requires_grad = True
    for param in model.classifier.parameters():
        param.requires_grad = True
    train(model)
    return model

headtuned_model = make_headtuned_model()
# Test — should show 592,130
print("Trainable params:", num_trainable_parameters(headtuned_model))
make_trainer(headtuned_model).save_model("headtuned")

In [ ]:
#Task 4.03: Extracting layers
def extract(model):
    named_layers = {}
    for i in range(6):
        for proj in ["q_lin", "v_lin"]:
            name = f"distilbert.transformer.layer.{i}.attention.{proj}"
            named_layers[name] = model.get_submodule(name)
    return named_layers

# Test — should print 7,087,104
extracted = extract(pretrained_model)
total = sum(p.numel() for l in extracted.values() for p in l.parameters())
print("Extracted params:", total)


In [ ]:
#Task 4.04: Replacing layers
import torch.nn as nn

def replace(model, named_layers):
    for name, new_layer in named_layers.items():
        parent_name, attr = name.rsplit(".", 1)
        parent = model.get_submodule(parent_name)
        setattr(parent, attr, new_layer)
    return model

def clone_linear(original):
    out_f, in_f = original.weight.shape
    copy = nn.Linear(in_f, out_f)
    copy.load_state_dict(original.state_dict())
    return copy

original_layers = extract(finetuned_model)
random_layers = {n: nn.Linear(l.in_features, l.out_features)
                 for n, l in original_layers.items()}
replace(finetuned_model, random_layers)
print("After random replacement:", evaluate_model(finetuned_model))  
replace(finetuned_model, original_layers)
print("After restoration:", evaluate_model(finetuned_model))         


In [ ]:
#Task 4.05: Low-rank matrix approximation
def approximate(matrix, rank):
    U, S, Vh = torch.linalg.svd(matrix, full_matrices=False)
    return U[:, :rank] @ torch.diag(S[:rank]) @ Vh[:rank, :]

# Test 
orig = torch.rand(768, 8) @ torch.rand(8, 384)
approx = approximate(orig, 8)
torch.dist(orig, approx)
print("Distance:", torch.dist(orig, approx).item())


In [ ]:
#Task 4.06: Approximated fine-tuned model (version 1)
def make_approximated_model_1(rank):
    model = AutoModelForSequenceClassification.from_pretrained("headtuned")
    for name, layer in extract(finetuned_model).items():
        new_layer = clone_linear(layer)
        new_layer.weight.data = approximate(layer.weight.data, rank)
        replace(model, {name: new_layer})
    return model

approximated_model_1 = make_approximated_model_1(768)
evaluate_model(approximated_model_1)

In [ ]:
#Task 4.07: Approximated fine-tuned model (version 2)
def make_approximated_model_2(rank):
    model = AutoModelForSequenceClassification.from_pretrained("headtuned")
    pt_layers = extract(pretrained_model)
    ft_layers = extract(finetuned_model)
    for name in ft_layers:
        W0 = pt_layers[name].weight.data
        dW = ft_layers[name].weight.data - W0
        new_layer = clone_linear(ft_layers[name])
        new_layer.weight.data = W0 + approximate(dW, rank)
        replace(model, {name: new_layer})
    return model

approximated_model_2 = make_approximated_model_2(768)
evaluate_model(approximated_model_2)

approximated_model_2 = make_approximated_model_2(3)
evaluate_model(approximated_model_2)

In [ ]:
#Task 4.08: Implement the adapter
import torch
import torch.nn as nn


# Task 4.08
class LoRA(nn.Module):
    def __init__(self, pretrained, rank=12, alpha=24):
        super().__init__()
        self.pretrained = pretrained
        self.rank  = rank
        self.alpha = alpha
        in_f  = pretrained.in_features
        out_f = pretrained.out_features
        self.A = nn.Parameter(torch.randn(in_f, rank))
        self.B = nn.Parameter(torch.zeros(rank, out_f))

    def forward(self, x):
        return self.pretrained(x) + (x @ self.A @ self.B) * (self.alpha / self.rank)

In [ ]:
#Task 4.09: Inject the adapter into the pre-trained model
def make_lora_model(rank):
    alpha = 2 * rank
    model = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=2
    )
    for param in model.parameters():
        param.requires_grad = False
    pretrained_qv = extract(model)
    lora_layers = {n: LoRA(l, rank=rank, alpha=alpha)
                   for n, l in pretrained_qv.items()}
    replace(model, lora_layers)
    for lora in lora_layers.values():
        lora.A.requires_grad = True
        lora.B.requires_grad = True
    for param in model.pre_classifier.parameters():
        param.requires_grad = True
    for param in model.classifier.parameters():
        param.requires_grad = True
    train(model)
    return model

lora_model = make_lora_model(6)
evaluate_model(lora_model)


In [ ]:
#Task 4.10: Comparing with a non-neural classifier

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

train_df = pd.read_csv("train.csv")
eval_df  = pd.read_csv("eval.csv")

vec = TfidfVectorizer(max_features=20_000, ngram_range=(1, 2))
X_train = vec.fit_transform(train_df["review"])
X_eval  = vec.transform(eval_df["review"])

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, train_df["label"])

acc = accuracy_score(eval_df["label"], clf.predict(X_eval))
print(f"Logistic Regression accuracy: {acc:.4f}")